In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D9 — Estatística do Movimento Fisiológico da População
#      de Portugal — Ano de 1925
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-por tesseract-ocr-fra -qq
!pip install -q pymupdf pdf2image pytesseract pillow pandas

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import math
import re

import fitz
import pandas as pd
import pytesseract

from pdf2image import convert_from_path
from PIL import ImageFilter, ImageOps

DOCUMENT_ID = "D9"

DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População "
    "de Portugal — Ano de 1925"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "300-DPI rendering with Tesseract por+fra OCR, "
    "deterministic orientation/PSM selection and "
    "page-aware Markdown conversion"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1"
EXPECTED_PAGE_COUNT = 8

EXPECTED_RECORD_COUNT = 19

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_TOPICS = {
    "Publication metadata": [
        "Title",
        "Reference year",
        "Publication year",
        "Publisher",
        "Institution"
    ],
    "Index entry": [
        "Tabela I",
        "Tabela II",
        "Tabela III",
        "Tabela XIV",
        "Tabela LVIII",
        "Tabela LIX"
    ],
    "Statistical value": [
        "Portugal area",
        "Portugal population 1911",
        "Portugal population 1920",
        "Portugal density 1920",
        "Portugal average annual population growth"
    ],
    "Document structure": [
        "Rotated table",
        "Bilingual headings",
        "Historical typography"
    ]
}

ROTATION_CANDIDATES = {
    7: [90, -90],
    8: [90, -90]
}

RENDER_DPI = 300

OUTPUT_DIR = Path("outputs_D9_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTICS_PATH = OUTPUT_DIR / "D9_branch_B_source_diagnostics.json"
OCR_RESULTS_PATH = OUTPUT_DIR / "D9_branch_B_ocr_results.csv"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D9_branch_B_structural_markdown.md"
CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D9_branch_B_conversion_integrity.json"
REPRESENTATION_PATH = OUTPUT_DIR / "D9_branch_B_representation.json"
PROMPT_PATH = OUTPUT_DIR / "D9_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D9_branch_B_experiment_metadata_pre.json"
RAW_RESPONSE_PATH = OUTPUT_DIR / "D9_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D9_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = OUTPUT_DIR / "D9_branch_B_technical_diagnostics.json"
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D9_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D9_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Frozen Stage 1 records:", EXPECTED_RECORD_COUNT)


In [ ]:
# ============================================================
# 1. Upload and verify the exact original D9 scanned PDF
# ============================================================

uploaded = files.upload()

pdf_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError("Upload exactly one original D9 PDF.")

SOURCE_PATH = pdf_paths[0]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D9 source format.")

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D9 PDF does not match the frozen Stage 1 source identity."
    )

pdf_document = fitz.open(SOURCE_PATH)
PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = PAGE_COUNT == EXPECTED_PAGE_COUNT

if not PAGE_COUNT_VALID:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

native_page_rows = []

for page_number, page in enumerate(pdf_document, start=1):
    native_text = page.get_text("text") or ""

    native_page_rows.append({
        "Page Number": page_number,
        "Native Character Count": len(native_text),
        "Native Word Count": len(native_text.split()),
        "Native Text Extractable": bool(native_text.strip()),
        "Width": float(page.rect.width),
        "Height": float(page.rect.height),
        "PDF Rotation": int(page.rotation)
    })

native_page_df = pd.DataFrame(native_page_rows)

TOTAL_NATIVE_CHARACTERS = int(
    native_page_df["Native Character Count"].sum()
)

TEXT_EXTRACTABLE = bool(
    TOTAL_NATIVE_CHARACTERS > 100
)

OCR_REQUIRED = not TEXT_EXTRACTABLE

if not OCR_REQUIRED:
    raise ValueError(
        "The frozen D9 source is expected to be image-based. "
        "Native text was unexpectedly detected."
    )

print("Source SHA-256:", SOURCE_SHA256)
print("Page count:", PAGE_COUNT)
print("Native characters:", TOTAL_NATIVE_CHARACTERS)
print("OCR required:", OCR_REQUIRED)

display(native_page_df)


In [ ]:
# ============================================================
# 2. Render all eight physical PDF pages
# ============================================================

rendered_pages = convert_from_path(
    str(SOURCE_PATH),
    dpi=RENDER_DPI,
    fmt="png"
)

if len(rendered_pages) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        "Rendered page count does not match the original PDF page count."
    )

image_metadata_rows = []

for page_number, image in enumerate(rendered_pages, start=1):

    orientation = (
        "landscape"
        if image.width > image.height
        else "portrait"
    )

    image_metadata_rows.append({
        "Page Number": page_number,
        "Image Width": image.width,
        "Image Height": image.height,
        "Rendered Orientation": orientation,
        "Rendered DPI": RENDER_DPI
    })

image_metadata_df = pd.DataFrame(image_metadata_rows)

print("Rendered pages:", len(rendered_pages))
display(image_metadata_df)


In [ ]:
# ============================================================
# 3. OCR conversion with deterministic orientation and
#    page-segmentation handling
# ============================================================

def prepare_for_ocr(image, rotation=0):
    """
    Prepare a rendered page for OCR.

    The operation is deterministic and does not alter source
    semantics:
    - optional rotation;
    - grayscale conversion;
    - contrast enhancement;
    - sharpening.
    """

    working = image.copy()

    if rotation:
        working = working.rotate(
            rotation,
            expand=True,
            fillcolor="white"
        )

    working = ImageOps.grayscale(
        working
    )

    working = ImageOps.autocontrast(
        working
    )

    working = working.filter(
        ImageFilter.SHARPEN
    )

    return working


def useful_token_count(text):
    """
    Count word-like and numeric OCR tokens.
    """

    return len(
        re.findall(
            r"[A-Za-zÀ-ÿ]{2,}|"
            r"\d+(?:[.,]\d+)?",
            str(text)
        )
    )


def numeric_token_count(text):
    """
    Count numeric-looking tokens.
    """

    return len(
        re.findall(
            r"(?<!\w)"
            r"[+-]?"
            r"\d[\d. ]*"
            r"(?:,\d+)?",
            str(text)
        )
    )


def normalise_ocr_search(text):
    """
    Normalised copy used only to evaluate OCR candidates.
    Does not modify the OCR text retained for model input.
    """

    text = str(text).casefold()

    text = " ".join(
        text.split()
    )

    return text


# ------------------------------------------------------------
# OCR segmentation modes
# ------------------------------------------------------------
#
# PSM 6: Assume one uniform block of text.
#
# PSM 11: Sparse-text recognition; useful for title/imprint pages
#   and visually distributed historical layouts.
#
# PSM 3: Fully automatic page segmentation.
#
# All candidates use the same OCR engine/languages.
# ------------------------------------------------------------

OCR_PSM_CANDIDATES = [
    3,
    6,
    11
]


# ------------------------------------------------------------
# Page-specific source-structure markers
# ------------------------------------------------------------

PAGE_STRUCTURE_MARKERS = {

    1: [
        r"movimento\s+fisiol[oó]gico",
        r"\b1925\b",
        r"\b1929\b"
    ],

    2: [
        r"movimento\s+fisiol[oó]gico",
        r"popula[cç][aã]o"
    ],

    3: [
        r"[íi]ndice",
        r"tabela\s+i\b",
        r"tabela\s+ii\b"
    ],

    4: [
        r"cidade\s+de\s+lisboa",
        r"cidade\s+do\s+p[oô]rto"
    ],

    5: [
        r"taxas\s+demogr[aá]ficas",
        r"tabela\s+lviii",
        r"tabela\s+lix"
    ],

    6: [
        r"tabela\s+i\b",
        r"\bportugal\b",
        r"\b1911\b",
        r"\b1920\b"
    ],

    7: [
        r"tabela\s+ii\b",
        r"\b1911\b"
    ],

    8: [
        r"tabela\s+ii\b",
        r"\b1911\b"
    ]
}


def structural_marker_hits(
    page_number,
    text
):
    """
    Count page-specific structural markers recovered by OCR.
    """

    patterns = PAGE_STRUCTURE_MARKERS.get(
        page_number,
        []
    )

    return sum(
        bool(
            re.search(
                pattern,
                str(text),
                flags=re.IGNORECASE
            )
        )
        for pattern in patterns
    )


ocr_rows = []
ocr_page_texts = {}

OCR_CANDIDATE_AUDIT = []


for page_number, image in enumerate(
    rendered_pages,
    start=1
):

    # --------------------------------------------------------
    # Orientation candidates
    # --------------------------------------------------------

    candidate_rotations = (
        ROTATION_CANDIDATES.get(
            page_number,
            [0]
        )
    )

    candidates = []


    # --------------------------------------------------------
    # Test orientation × PSM combinations
    # --------------------------------------------------------

    for rotation in candidate_rotations:

        prepared = prepare_for_ocr(
            image,
            rotation=rotation
        )

        for psm in OCR_PSM_CANDIDATES:

            ocr_text = (
                pytesseract.image_to_string(
                    prepared,
                    lang="por+fra",
                    config=(
                        "--oem 1 "
                        f"--psm {psm}"
                    )
                )
            )

            useful_tokens = (
                useful_token_count(
                    ocr_text
                )
            )

            numeric_tokens = (
                numeric_token_count(
                    ocr_text
                )
            )

            marker_hits = (
                structural_marker_hits(
                    page_number,
                    ocr_text
                )
            )

            candidate = {
                "page_number":
                    page_number,

                "rotation":
                    rotation,

                "psm":
                    psm,

                "text":
                    ocr_text,

                "character_count":
                    len(ocr_text),

                "word_count":
                    len(
                        ocr_text.split()
                    ),

                "useful_token_count":
                    useful_tokens,

                "numeric_token_count":
                    numeric_tokens,

                "structural_marker_hits":
                    marker_hits
            }

            candidates.append(
                candidate
            )

            OCR_CANDIDATE_AUDIT.append({
                key: value
                for key, value
                in candidate.items()
                if key != "text"
            })


    # --------------------------------------------------------
    # Deterministic candidate selection
    # --------------------------------------------------------

    best_candidate = max(
        candidates,
        key=lambda item: (
            item[
                "structural_marker_hits"
            ],
            item[
                "useful_token_count"
            ],
            item[
                "character_count"
            ],
            -item[
                "psm"
            ]
        )
    )


    # --------------------------------------------------------
    # Preserve the selected raw OCR text
    # --------------------------------------------------------

    ocr_page_texts[
        page_number
    ] = best_candidate[
        "text"
    ]


    ocr_rows.append({

        "Page Number":
            page_number,

        "Candidate Rotations":
            candidate_rotations,

        "Selected Rotation":
            best_candidate[
                "rotation"
            ],

        "Selected PSM":
            best_candidate[
                "psm"
            ],

        "Structural Marker Hits":
            best_candidate[
                "structural_marker_hits"
            ],

        "OCR Character Count":
            best_candidate[
                "character_count"
            ],

        "OCR Word Count":
            best_candidate[
                "word_count"
            ],

        "Useful Token Count":
            best_candidate[
                "useful_token_count"
            ],

        "Numeric Token Count":
            best_candidate[
                "numeric_token_count"
            ],

        "OCR Text":
            best_candidate[
                "text"
            ]
    })


# ------------------------------------------------------------
# Preserve selected OCR results
# ------------------------------------------------------------

ocr_results_df = pd.DataFrame(
    ocr_rows
)


ocr_results_df.to_csv(
    OCR_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)


ocr_candidate_audit_df = pd.DataFrame(
    OCR_CANDIDATE_AUDIT
)


FULL_OCR_TEXT = "\n".join(
    ocr_page_texts[
        page_number
    ]
    for page_number
    in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
)


print(
    "OCR pages completed:",
    len(ocr_page_texts)
)


display(
    ocr_results_df[
        [
            "Page Number",
            "Selected Rotation",
            "Selected PSM",
            "Structural Marker Hits",
            "OCR Character Count",
            "OCR Word Count",
            "Useful Token Count",
            "Numeric Token Count"
        ]
    ]
)

In [ ]:
# ============================================================
# 4. Create the complete page-aware structural Markdown
# ============================================================

def lightly_structure_ocr_text(text):
    """
    Preserve OCR wording and line order while exposing only
    deterministic heading-like cues already represented in the OCR.
    No spelling/value correction is performed.
    """

    text = (
        str(text)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    output_lines = []

    for raw_line in text.splitlines():

        stripped = raw_line.strip()

        if not stripped:
            output_lines.append("")
            continue

        if re.fullmatch(
            r"(ÍNDICE|INDICE|TABLE DES MATI[ÈE]RES)",
            stripped,
            flags=re.IGNORECASE
        ):
            output_lines.append(
                f"### {stripped}"
            )

        elif re.fullmatch(
            r"TABELA\s+[IVXLCDM]+.*",
            stripped,
            flags=re.IGNORECASE
        ):
            output_lines.append(
                f"### {stripped}"
            )

        else:
            output_lines.append(
                raw_line.rstrip()
            )

    return re.sub(
        r"\n{3,}",
        "\n\n",
        "\n".join(output_lines)
    ).strip()


markdown_parts = [
    "# D9 — Estatística do Movimento Fisiológico da População de Portugal",
    "",
    "> Complete OCR-based structural conversion of the supplied eight-page scanned PDF.",
    "> All physical source pages are retained.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):

    selected_rotation = int(
        ocr_results_df.loc[
            ocr_results_df["Page Number"] == page_number,
            "Selected Rotation"
        ].iloc[0]
    )

    markdown_parts.extend([
        f"## Source Page {page_number}",
        ""
    ])

    if page_number in ROTATION_CANDIDATES:
        markdown_parts.extend([
            (
                "> Structural conversion note: this physical source page "
                f"required a {selected_rotation}-degree rotation for OCR reading."
            ),
            ""
        ])

    markdown_parts.extend([
        lightly_structure_ocr_text(
            ocr_page_texts[page_number]
        ),
        ""
    ])

STRUCTURAL_MARKDOWN = (
    "\n".join(markdown_parts)
    .rstrip()
    + "\n"
)

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)

print("Structural Markdown:", STRUCTURAL_MARKDOWN_PATH)
print("Representation SHA-256:", STRUCTURAL_MARKDOWN_SHA256)
print("Characters:", len(STRUCTURAL_MARKDOWN))


In [ ]:
# ============================================================
# 5. Conversion-integrity verification
# ============================================================


# ------------------------------------------------------------
# 5.1 Page-boundary checks
# ------------------------------------------------------------

page_boundary_checks = {
    str(page_number):
        f"## Source Page {page_number}"
        in STRUCTURAL_MARKDOWN
    for page_number
    in range(1, EXPECTED_PAGE_COUNT + 1)
}


# ------------------------------------------------------------
# 5.2 Non-empty OCR checks
# ------------------------------------------------------------

page_nonempty_checks = {
    str(page_number):
        bool(
            ocr_page_texts[
                page_number
            ].strip()
        )
    for page_number
    in range(1, EXPECTED_PAGE_COUNT + 1)
}


# ------------------------------------------------------------
# 5.3 Page-specific material-recovery checks
# ------------------------------------------------------------

MINIMUM_USEFUL_TOKENS_BY_PAGE = {
    1: 10,
    2: None,
    4: 10,
    5: 10,
    6: 10,
    7: 10,
    8: 10
}


observed_page_token_counts = {}

minimum_page_token_checks = {}


for _, row in ocr_results_df.iterrows():

    page_number = int(
        row["Page Number"]
    )

    observed_tokens = int(
        row["Useful Token Count"]
    )

    required_tokens = (
        MINIMUM_USEFUL_TOKENS_BY_PAGE[
            page_number
        ]
    )

    observed_page_token_counts[
        str(page_number)
    ] = observed_tokens


    if required_tokens is None:

        minimum_page_token_checks[
            str(page_number)
        ] = page_nonempty_checks[
            str(page_number)
        ]

    else:

        minimum_page_token_checks[
            str(page_number)
        ] = (
            observed_tokens
            >= required_tokens
        )


# ------------------------------------------------------------
# 5.4 Critical source-marker checks
# ------------------------------------------------------------

MARKER_PATTERNS = {

    "publication_title":
        r"movimento\s+fisiol[oó]gico",

    "reference_year":
        r"\b1925\b",

    "publication_year":
        r"\b1929\b",

    "index_heading":
        r"[íi]ndice|table\s+des\s+mati",

    "tabela_i":
        r"tabela\s+i\b",

    "tabela_ii":
        r"tabela\s+i[\s|l]*i\b",

    "portugal":
        r"\bportugal\b",

    "1911":
        r"\b1911\b",

    "1920":
        r"\b1920\b"
}


marker_status = {
    marker:
        bool(
            re.search(
                pattern,
                STRUCTURAL_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for marker, pattern
    in MARKER_PATTERNS.items()
}


# ------------------------------------------------------------
# 5.5 Selected rotations for pages 7 and 8
# ------------------------------------------------------------

selected_rotations = {
    str(page_number):
        int(
            ocr_results_df.loc[
                ocr_results_df[
                    "Page Number"
                ] == page_number,
                "Selected Rotation"
            ].iloc[0]
        )
    for page_number
    in [7, 8]
}


rotated_pages_valid = all(
    rotation in [90, -90]
    for rotation
    in selected_rotations.values()
)


# ------------------------------------------------------------
# 5.6 Selected PSM diagnostics
# ------------------------------------------------------------

if "Selected PSM" in ocr_results_df.columns:

    selected_psm = {
        str(page_number):
            int(
                ocr_results_df.loc[
                    ocr_results_df[
                        "Page Number"
                    ] == page_number,
                    "Selected PSM"
                ].iloc[0]
            )
        for page_number
        in range(1, EXPECTED_PAGE_COUNT + 1)
    }

else:

    selected_psm = None


# ------------------------------------------------------------
# 5.7 Conversion-integrity object
# ------------------------------------------------------------

CONVERSION_INTEGRITY = {

    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "observed_page_count":
        PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "native_text_characters":
        TOTAL_NATIVE_CHARACTERS,

    "ocr_required":
        OCR_REQUIRED,

    "render_dpi":
        RENDER_DPI,

    "ocr_languages": [
        "por",
        "fra"
    ],

    "all_source_pages_retained":
        all(
            page_boundary_checks.values()
        ),

    "page_boundary_checks":
        page_boundary_checks,

    "page_nonempty_checks":
        page_nonempty_checks,

    "all_ocr_pages_nonempty":
        all(
            page_nonempty_checks.values()
        ),

    "minimum_useful_tokens_by_page":
        MINIMUM_USEFUL_TOKENS_BY_PAGE,

    "observed_useful_tokens_by_page":
        observed_page_token_counts,

    "minimum_page_token_checks":
        minimum_page_token_checks,

    "all_pages_materially_recovered":
        all(
            minimum_page_token_checks.values()
        ),

    "marker_status":
        marker_status,

    "all_critical_markers_preserved":
        all(
            marker_status.values()
        ),

    "selected_rotations_pages_7_8":
        selected_rotations,

    "rotated_pages_valid":
        rotated_pages_valid,

    "selected_psm_by_page":
        selected_psm,

    "conversion_method":
        CONVERSION_METHOD,

    "ocr_conversion_applied":
        True,

    "structural_conversion_applied":
        True,

    "conversion_fallback_used":
        False,

    "complete_source_document_retained":
        True,

    "scope_filtering_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "ocr_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "normalisation_applied":
        False,

    "conversion_integrity_passed":
        all([
            SOURCE_HASH_MATCH,
            PAGE_COUNT_VALID,
            OCR_REQUIRED,

            all(
                page_boundary_checks.values()
            ),

            all(
                page_nonempty_checks.values()
            ),

            all(
                minimum_page_token_checks.values()
            ),

            all(
                marker_status.values()
            ),

            rotated_pages_valid
        ])
}


# ------------------------------------------------------------
# 5.8 Save conversion-integrity report
# ------------------------------------------------------------

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5.9 Source diagnostics
# ------------------------------------------------------------

SOURCE_DIAGNOSTICS = {

    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_page_count":
        PAGE_COUNT,

    "native_text_characters":
        TOTAL_NATIVE_CHARACTERS,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "render_dpi":
        RENDER_DPI,

    "ocr_languages": [
        "Portuguese",
        "French"
    ],

    "selected_psm_by_page":
        selected_psm,

    "rotated_pages": [
        7,
        8
    ],

    "selected_rotations":
        selected_rotations,

    "structural_markdown_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "structural_markdown_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY[
            "conversion_integrity_passed"
        ]
}


SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5.10 Display result
# ------------------------------------------------------------

print(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


# ------------------------------------------------------------
# 5.11 Stop only if integrity actually failed
# ------------------------------------------------------------

if not CONVERSION_INTEGRITY[
    "conversion_integrity_passed"
]:
    raise ValueError(
        "D9 Branch B OCR structural conversion "
        "failed integrity checks."
    )

In [ ]:
# ============================================================
# 6. Preserve Branch B representation metadata
# ============================================================

REPRESENTATION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Complete scanned PDF converted to page-aware OCR structural Markdown",
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": EXPECTED_PAGE_COUNT,
    "structural_conversion_applied": True,
    "conversion_method":
        CONVERSION_METHOD,
    "ocr_applied": True,
    "ocr_languages": ["por", "fra"],
    "complete_source_document_retained": True,
    "page_boundaries_made_explicit": True,
    "rotated_source_pages": [7, 8],
    "orientation_selection_method":
        (
            "Deterministic selection across orientation and PSM "
            "candidates, prioritising page-specific structural-marker "
            "recovery, then useful-token recovery and character count"
        ),
    "scope_enforced_by_prompt_not_representation_filtering": True,
    "conversion_fallback_used": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "ocr_spelling_correction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(REPRESENTATION, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 7. Controlled Branch B prompt
# ============================================================

BRANCH_B_PROMPT = """You are an information extraction assistant.

Extract the predefined bibliographic, index, statistical and structural
records represented within the defined scope of the attached OCR-based
structural Markdown representation of the original historical scanned
PDF publication:

“Estatística do Movimento Fisiológico da População de Portugal —
Ano de 1925”.

Treat the attached structural Markdown representation as the only
source of information.

The supplied representation corresponds to the complete eight physical
pages of the scanned PDF subset. Do not infer information from omitted
printed pages of the larger historical publication.

Use exactly these record fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Use exactly one of these Category values:

- Publication metadata
- Index entry
- Statistical value
- Document structure


1. Publication metadata

Using the cover and imprint represented from physical PDF page 1,
extract one record for each of these predefined topics:

- Title
- Reference year
- Publication year
- Publisher
- Institution

Preserve the represented Portuguese title and institution wording.

Keep the statistical reference year distinct from the printed
publication year.

Do not infer bibliographic information that is not explicitly
represented in the supplied source representation.


2. Selected index entries

Using the index content represented from the supplied PDF, extract one
record for each of these selected table identifiers:

- Tabela I
- Tabela II
- Tabela III
- Tabela XIV
- Tabela LVIII
- Tabela LIX

For each selected index entry:

- preserve the Portuguese table identifier;
- preserve the Portuguese table description as represented by OCR;
- preserve the represented printed page number or page range in the
  Source Location where recoverable;
- preserve any explicitly associated period;
- do not create a separate record from the accompanying French
  translation.

The first four selected entries originate from physical PDF page 3.
The final two selected entries originate from physical PDF page 5.


3. Selected statistical values from Tabela I

From the Portugal row of Tabela I represented from physical PDF page 6,
extract one record for each of these predefined indicators:

- Portugal area
- Portugal population 1911
- Portugal population 1920
- Portugal density 1920
- Portugal average annual population growth

Read each value directly from the represented Portugal row and its
corresponding OCR-recovered column context.

Preserve the represented numeric scale.

Do not calculate, derive, estimate, interpolate, rescale, correct or
convert any value.

Do not extract district-level, city-level, sex-specific or other
Tabela I observations outside this predefined scope.


4. Document structure

Create one source-grounded structural record for each of these
predefined topics:

- Rotated table
- Bilingual headings
- Historical typography

For “Rotated table”, use only structural evidence explicitly retained
in the representation concerning the orientation and multi-page
presentation of Tabela II.

For “Bilingual headings”, describe the represented relationship between
the Portuguese headings and their accompanying translated headings.

For “Historical typography”, describe only characteristics that remain
explicitly supported by the OCR-converted representation. Do not invent
visual details that are not preserved by the conversion.

Do not infer any quantitative values from Tabela II.


Field rules:

Category:
- Use exactly one of the four category labels defined above.

Topic:
- For each record, use the corresponding predefined topic label from
  the scope above.

Description:
- Provide a concise source-grounded description of the represented
  record.
- Preserve Portuguese source wording where the record is a publication
  title or table description.
- Do not add external interpretation.

Value:
- Use a JSON number when the source representation explicitly provides
  a numeric value.
- Use a JSON string when the represented value is textual.
- Use null only when no separate Value is represented.
- Preserve the represented printed scale.
- Do not calculate, infer, derive, convert, repair or modernise values.

Unit:
- Preserve the explicitly associated measurement unit where recoverable.
- Use null when no explicit unit applies.
- Do not invent units.

Reporting Period:
- Preserve an explicitly associated reference year or period.
- Keep the statistical reference year and publication year distinct.
- Preserve the census periods associated with the selected statistical
  observations where explicitly represented.
- Use null when no explicit reporting period applies.

Source Location:
- Use physical PDF page references exposed by the Markdown page
  boundaries.
- Where recoverable, also preserve the represented printed page or page
  range for an index entry.
- Use concise locations such as:
  “PDF page 1 — Cover”
  “PDF page 1 — Imprint”
  “PDF page 3 — Índice; printed page ...”
  “PDF page 5 — Índice; printed page ...”
  “PDF page 6 — Tabela I, Portugal row”
  “PDF pages 7–8 — Tabela II”


Additional extraction rules:

- Use only information explicitly represented in the attached
  structural Markdown.
- Preserve Portuguese titles, table identifiers, historical spelling
  and punctuation where recoverable.
- Do not silently correct OCR wording or numerical values.
- Do not treat French translations as separate duplicate records.
- Do not use the original PDF, external OCR, external knowledge, or
  information outside the attached representation.
- Do not calculate, infer, derive, estimate or reconstruct missing
  information.
- Do not infer age-by-sex or other quantitative observations from
  Tabela II.
- Do not extract observations outside the predefined scope.
- Ignore Markdown syntax and conversion labels except as structural
  cues.
- Verify that every item within the defined scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D9",
  "branch": "B",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
""".strip()

PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print("Prompt saved:", PROMPT_PATH.name)
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# ============================================================
# 8. Create pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "llm_input_representation": LLM_INPUT_REPRESENTATION,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "conversion_method":
        CONVERSION_METHOD,
    "ocr_applied_for_model_input": True,
    "ocr_languages": ["por", "fra"],
    "complete_source_document_retained": True,
    "scope_filtering_applied": False,
    "page_boundaries_made_explicit": True,
    "rotated_source_pages": [7, 8],
    "conversion_fallback_used": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "ocr_spelling_correction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "content_validation_performed":
        False,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "source_diagnostics_file": SOURCE_DIAGNOSTICS_PATH.name,
    "ocr_results_file": OCR_RESULTS_PATH.name,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format":
        "JSON object with document_id, branch and records",
    "execution_environment": "Independent ChatGPT conversation",
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_METADATA_PRE, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 9. Download Branch B model-input artefacts
# ============================================================

for path in [
    SOURCE_DIAGNOSTICS_PATH,
    OCR_RESULTS_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D9_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D9_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original PDF, Stage 1 reference dataset, "
    "Branch A outputs, or expected record/category counts.\n"
    "5. Save the first complete response exactly as returned as TXT.\n"
    "6. Do not correct, repair, reorder, or regenerate the response."
)


In [ ]:
# ============================================================
# 10. Upload and preserve the untouched model response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete D9 Branch B response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]

RAW_RESPONSE_TEXT = UPLOADED_RAW_RESPONSE_PATH.read_text(
    encoding="utf-8"
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError(
        "The uploaded D9 Branch B response is empty."
    )

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print("Raw response preserved:", RAW_RESPONSE_PATH.name)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 11. Parsing without repairing the model response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )
    valid_json = True

except json.JSONDecodeError as error:
    json_parsing_error = str(error)


top_level_object_valid = (
    valid_json
    and isinstance(parsed_response, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get("records"),
        list
    )
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

print("Valid JSON:", valid_json)
print("JSON parsing error:", json_parsing_error)
print("Records evaluable:", records_evaluable)
print("Observed record count:", observed_record_count)


In [ ]:
# ============================================================
# 12. Record-schema and field-type checks
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    observed_fields = list(
        record.keys()
    )

    missing_fields = [
        field
        for field in EXPECTED_FIELDS
        if field not in record
    ]

    extra_fields = [
        field
        for field in observed_fields
        if field not in EXPECTED_FIELDS
    ]

    field_order_correct = (
        observed_fields == EXPECTED_FIELDS
    )

    if (
        missing_fields
        or extra_fields
        or not field_order_correct
    ):
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields,
            "field_order_correct": field_order_correct,
            "observed_fields": observed_fields
        })

    for field in STRING_OR_NULL_FIELDS:

        value = record.get(field)

        if (
            value is not None
            and not isinstance(value, str)
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__,
                "expected_type": "string or null"
            })

    value = record.get("Value")

    if (
        isinstance(value, bool)
        or (
            value is not None
            and not isinstance(
                value,
                (str, int, float)
            )
        )
    ):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Value",
            "observed_type": type(value).__name__,
            "expected_type":
                "string, number or null"
        })

    for field in MANDATORY_CONTENT_FIELDS:

        value = record.get(field)

        if value is None or value == "":
            missing_mandatory_values.append({
                "record_index": record_index,
                "field": field
            })


record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)

records_with_type_issues = (
    len({
        issue["record_index"]
        for issue in field_type_issues
    })
    if records_evaluable
    else None
)

print("Record schema valid:", record_schema_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)


In [ ]:
# ============================================================
# 13. Content/scope diagnostics — separation from schema validity
# ============================================================

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_records = [
        list(key)
        for key, count in duplicate_counter.items()
        if count > 1
    ]

    duplicate_complete_record_count = len(
        duplicate_records
    )

    numeric_value_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Value"),
                (int, float)
            )
            and not isinstance(
                record.get("Value"),
                bool
            )
        )
    )

    text_value_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Value"),
                str
            )
        )
    )

    null_value_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Value") is None
        )
    )

    observed_topics_by_category = {
        category: sorted([
            record.get("Topic")
            for record in extracted_records
            if (
                isinstance(record, dict)
                and record.get("Category") == category
            )
        ])
        for category in EXPECTED_CATEGORY_COUNTS
    }

    expected_topic_presence = {
        category: {
            topic: any(
                isinstance(record, dict)
                and record.get("Category") == category
                and record.get("Topic") == topic
                for record in extracted_records
            )
            for topic in expected_topics
        }
        for category, expected_topics
        in EXPECTED_TOPICS.items()
    }

    all_expected_topics_present = all(
        status
        for category_status
        in expected_topic_presence.values()
        for status
        in category_status.values()
    )


    topic_to_records = {}

    for record in extracted_records:
        if isinstance(record, dict):
            topic_to_records.setdefault(
                record.get("Topic"),
                []
            ).append(record)

    reference_year_records = topic_to_records.get(
        "Reference year",
        []
    )

    publication_year_records = topic_to_records.get(
        "Publication year",
        []
    )

    year_distinction_preserved = (
        len(reference_year_records) == 1
        and len(publication_year_records) == 1
        and reference_year_records[0].get("Value")
            != publication_year_records[0].get("Value")
    )

    tabela_ii_quantitative_records = [
        record
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category") == "Statistical value"
            and isinstance(
                record.get("Source Location"),
                str
            )
            and "tabela ii" in
                record.get("Source Location").casefold()
        )
    ]

    no_tabela_ii_quantitative_extraction = (
        len(tabela_ii_quantitative_records) == 0
    )

else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_records = None
    duplicate_complete_record_count = None
    numeric_value_count = None
    text_value_count = None
    null_value_count = None
    observed_topics_by_category = None
    expected_topic_presence = None
    all_expected_topics_present = None
    year_distinction_preserved = None
    tabela_ii_quantitative_records = None
    no_tabela_ii_quantitative_extraction = None


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(missing_mandatory_values)
            if records_evaluable
            else None
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "expected_topic_presence":
        expected_topic_presence,

    "all_expected_topics_present":
        all_expected_topics_present,

    "year_distinction_preserved":
        year_distinction_preserved,

    "no_tabela_ii_quantitative_extraction":
        no_tabela_ii_quantitative_extraction
}

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 14. Determine technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),

        "branch":
            parsed_response.get("branch"),

        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True

    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 15. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_file":
        RAW_RESPONSE_PATH.name,
    "raw_response_sha256":
        RAW_RESPONSE_SHA256,
    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),
    "parsed_extraction_sha256":
        parsed_extraction_sha256,
    "json_valid":
        valid_json,
    "records_evaluable":
        records_evaluable,
    "observed_record_count":
        observed_record_count,
    "observed_category_counts":
        observed_category_counts,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "structurally_evaluable":
        bool(structurally_evaluable),
    "notes": (
        "Branch B converts the complete image-based D9 PDF to "
        "page-aware OCR structural Markdown. Pages 7 and 8 use "
        "deterministic orientation/PSM selection established from "
        "Stage 1 document characterisation. OCR wording and numeric "
        "content are not manually corrected. Expected Stage 1 counts "
        "are used only for post-extraction diagnostics and are not "
        "disclosed to the model. Content-level validation is performed "
        "separately in Validation B — D9."
    )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY[
            "conversion_integrity_passed"
        ],

    "ocr_applied":
        True,

    "structural_conversion_applied":
        True,

    "complete_source_document_retained":
        True,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "valid_json":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "scope_complete":
        record_count_valid,

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "all_expected_topics_present":
        all_expected_topics_present,

    "year_distinction_preserved":
        year_distinction_preserved,

    "no_tabela_ii_quantitative_extraction":
        no_tabela_ii_quantitative_extraction,

    "parsed_extraction_created":
        bool(structurally_evaluable),

    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D9."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 16. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    SOURCE_DIAGNOSTICS_PATH,
    OCR_RESULTS_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(
        PARSED_EXTRACTION_PATH
    )

print("Generated D9 Branch B files:")

for path in GENERATED_OUTPUTS:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )

for path in GENERATED_OUTPUTS:
    files.download(path)
